In [1]:
# building sample vector db

from langchain_chroma import Chroma



In [2]:
from langchain_community.document_loaders import TextLoader

from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [3]:
loader=TextLoader("speech.txt")
data=loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I\'ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That\'s it. No big deal. Just three stories.\n\nThe first story is about connecting the dots.\n\nI dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out?\n\nIt started before I was born. My biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were

In [4]:
text_splitter=CharacterTextSplitter(chunk_size=100,chunk_overlap=40)
splits=text_splitter.split_documents(data)

Created a chunk of size 310, which is longer than the specified 100
Created a chunk of size 164, which is longer than the specified 100
Created a chunk of size 835, which is longer than the specified 100
Created a chunk of size 746, which is longer than the specified 100
Created a chunk of size 422, which is longer than the specified 100
Created a chunk of size 600, which is longer than the specified 100
Created a chunk of size 803, which is longer than the specified 100


In [6]:
embeddings=OllamaEmbeddings(model="llama3.2:latest")
db=Chroma.from_documents(splits,embeddings)
db

In [13]:
# query

query="What advice does Steve Jobs give about following your intuition and trusting the future?"
docs=db.similarity_search(query)
docs[0].page_content


"Again, you can't connect the dots looking forward; you can only connect them looking backwards. So you have to trust that the dots will somehow connect in your future. You have to trust in something — your gut, destiny, life, karma, whatever. This approach has never let me down, and it has made all the difference in my life."

In [15]:
#saving to the local disk

vectordb=Chroma.from_documents(splits,embeddings,persist_directory="./chroma_db")

In [10]:
newdb=Chroma(persist_directory="./chroma_db",embedding_function=embeddings)

docs=newdb.similarity_search(query)
docs[2].page_content

"I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I've ever gotten to a college graduation. Today I want to tell you three stories from my life. That's it. No big deal. Just three stories."

In [16]:
## retriever option

retriever=vectordb.as_retriever()

In [17]:
retriever.invoke(query)[0].page_content

"Again, you can't connect the dots looking forward; you can only connect them looking backwards. So you have to trust that the dots will somehow connect in your future. You have to trust in something — your gut, destiny, life, karma, whatever. This approach has never let me down, and it has made all the difference in my life."

In [18]:
new_query="What lessons does Steve Jobs share about life and death?"
retriever.invoke(new_query)[0].page_content

"It wasn't all romantic. I didn't have a dorm room, so I slept on the floor in friends' rooms, I returned Coke bottles for the 5¢ deposits to buy food with, and I would walk the 7 miles across town every Sunday night to get one good meal a week at the Hare Krishna temple. I loved it. And much of what I stumbled into by following my curiosity and intuition turned out to be priceless later on. Let me give you one example:"